# Zero-Shot Inference — todos os datasets × variantes

Gera as inferências **zero-shot** (sem fine-tuning) para os 4 datasets e as variantes de modelo/pesos, salvando os artefatos já no layout consumido por `utils_eval.py` / `eval_*.ipynb`.

> Substitui o legado `inference_zero_shot.ipynb` (que cobria só VessMap/DRIVE com ResNet). O legado permanece intocado.

## Variantes (5) × datasets (4) = 20 inferências

| variant_key (pasta raw) | model_class | pesos | label (`model_type`) | channels | resize |
|---|---|---|---|---|---|
| `resnet18_vessshape` | resnet18_unet | VessShape (checkpoint) | `Zero-Shot VSUNet18` | default | por dataset |
| `resnet50_vessshape` | resnet50_unet | VessShape (checkpoint) | `Zero-Shot VSUNet50` | default | por dataset |
| `resnet18_imagenet` | resnet18_unet | ImageNet (encoder via `smp`) | `Zero-Shot IN-UNet18` | default | por dataset |
| `resnet50_imagenet` | resnet50_unet | ImageNet (encoder via `smp`) | `Zero-Shot IN-UNet50` | default | por dataset |
| `litemedsam` | litemedsam | LiteMedSAM (`lite_medsam.pth`) | `Zero-Shot LiteMedSAM` | **rgb** | **256** |

**`resize_size` por dataset (ResNet):** vessmap 256 · drive 288 · dca1 288 · octa2d 384.  LiteMedSAM usa **256** sempre (encoder TinyViT fixo).

**Pesos:** só as variantes *VessShape* leem checkpoint do drive externo. *ImageNet* pega os pesos do encoder pela própria `segmentation_models_pytorch` (`encoder_weights='imagenet'` + `skip_checkpoint_loading`); *LiteMedSAM* carrega `lite_medsam.pth` interno do pacote `torchtrainer`.

**Normalização ImageNet:** opcional via toggle `IMAGENET_NORMALIZE` (default `False`) — para avaliar a variante IN-UNet com e sem normalização.

> **Nota técnica:** `src.test.test` converte o dict de parâmetros em `argv` e dá `.split()` em cada valor (necessário p/ `resize_size '256 256'`). Como o caminho do projeto contém espaços (Google Drive), a saída é escrita por um **symlink sem espaços** (`/tmp/...`) que aponta para `zero_shot_inferences/` — os arquivos caem no diretório real.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, traceback

sys.path.insert(0, os.path.abspath('..'))  # project root -> import src.test
from src.test import test

import utils_eval as ue
from utils_eval import (
    ZERO_SHOT_VARIANTS, ZERO_SHOT_RAW_DIR, ZERO_SHOT_DIR,
    build_zero_shot_csv, ensure_output_dirs, load_dataset_results,
)

ensure_output_dirs()
print('variantes:', list(ZERO_SHOT_VARIANTS))
print('raw dir   :', ZERO_SHOT_RAW_DIR)
print('csv dir   :', ZERO_SHOT_DIR)

/home/wesleygalvao/anaconda3/envs/mestrado_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


variantes: ['resnet18_vessshape', 'resnet50_vessshape', 'resnet18_imagenet', 'resnet50_imagenet', 'litemedsam', 'litemedsam_normalized']
raw dir   : /home/wesleygalvao/Insync/wesleygalv@gmail.com/Google Drive - Shared with me/Mestrado - Visão Computacional/Pesquisa/Experimentos/fase_4/vess-shape-experiments/04_evaluation/zero_shot_inferences
csv dir   : /home/wesleygalvao/Insync/wesleygalv@gmail.com/Google Drive - Shared with me/Mestrado - Visão Computacional/Pesquisa/Experimentos/fase_4/vess-shape-experiments/04_evaluation/zero_shot


In [2]:
# --- Configuração ----------------------------------------------------------------
DEVICE = 'cuda:0'
IMAGENET_NORMALIZE = False   # toggle: avaliar IN-UNet zero-shot com normalização ImageNet

DATASETS_TO_RUN = ['vessmap', 'drive', 'dca1', 'octa2d']

# resize por dataset para as variantes ResNet (LiteMedSAM usa 256 sempre)
RESIZE_BY_DATASET = {'vessmap': 256, 'drive': 288, 'dca1': 288, 'octa2d': 384}

# Checkpoints pré-treinados em VessShape (única variante que lê pesos do drive)
_VS_ROOT = '/media/wesleygalvao/1_TB_LINUX/models_results/01_pre-trained_on_vess_shape'
CHECKPOINTS = {
    'resnet18_unet': f'{_VS_ROOT}/resnet18_ts:50k_bs-train:192_ep:1000_lr:0.01_lr-decay:0.0_wd:0.0_opt:adam_class1:0.10_FP16',
    'resnet50_unet': f'{_VS_ROOT}/resnet50_ts:50k_bs-train:96_ep:3000_lr:0.001_lr-decay:0.0_wd:0.0001_opt:adam_class1:0.10_FP16',
}

_DS_ROOT = '/media/wesleygalvao/1_TB_LINUX/Datasets/blood_vessels'
DATASET_PATHS = {
    'vessmap': f'{_DS_ROOT}/VessMAP',
    'drive':   f'{_DS_ROOT}/DRIVE',
    'dca1':    f'{_DS_ROOT}/DCA1',
    'octa2d':  f'{_DS_ROOT}/OCTA2D',
}

# Symlink sem espaços -> ZERO_SHOT_RAW_DIR (workaround p/ dict_to_argv que dá split em espaços)
RAW_LINK = '/tmp/vss_zero_shot_raw'
os.makedirs(ZERO_SHOT_RAW_DIR, exist_ok=True)
if os.path.islink(RAW_LINK) and os.path.realpath(RAW_LINK) != os.path.realpath(ZERO_SHOT_RAW_DIR):
    os.remove(RAW_LINK)
if not os.path.exists(RAW_LINK):
    os.symlink(ZERO_SHOT_RAW_DIR, RAW_LINK)
assert ' ' not in RAW_LINK
print('symlink:', RAW_LINK, '->', os.path.realpath(RAW_LINK))

symlink: /tmp/vss_zero_shot_raw -> /home/wesleygalvao/Insync/wesleygalv@gmail.com/Google Drive - Shared with me/Mestrado - Visão Computacional/Pesquisa/Experimentos/fase_4/vess-shape-experiments/04_evaluation/zero_shot_inferences


In [3]:
import subprocess
from torchtrainer.util.train_util import dict_to_argv

# Cada inferência roda como SUBPROCESSO de src/test.py (igual ao orquestrador few-shot).
# Motivo: o LiteMedSAM do torchtrainer usa encoders singletons a nível de módulo; após a 1a
# chamada o test.py move-os p/ GPU e a 2a chamada in-process quebra com device-mismatch
# (cpu vs cuda). Subprocesso garante estado fresco por run e isola falhas.
_TEST_PY = os.path.abspath(os.path.join('..', 'src', 'test.py'))  # caminho com espaços OK em argv-list
_CWD = os.path.abspath('.')
_POSITIONAL = ['dataset_path', 'dataset_class', 'model_class']


def run_zero_shot(variant_key, dataset, imagenet_normalize=None):
    """Roda uma inferência zero-shot (subprocesso) e salva em
    zero_shot_inferences/<variant_key>/inference_results_<dataset>/. Retorna o caminho real."""
    if imagenet_normalize is None:
        imagenet_normalize = IMAGENET_NORMALIZE
    meta = ZERO_SHOT_VARIANTS[variant_key]
    model_class, weights = meta['model_class'], meta['weights']
    resize = 256 if weights == 'litemedsam' else RESIZE_BY_DATASET[dataset]

    real_out = os.path.join(ZERO_SHOT_RAW_DIR, variant_key, f'inference_results_{dataset}')
    link_out = os.path.join(RAW_LINK, variant_key, f'inference_results_{dataset}')  # sem espaços
    os.makedirs(os.path.dirname(link_out), exist_ok=True)  # test.py faz mkdir() sem parents=True

    params = {
        'dataset_path': DATASET_PATHS[dataset],
        'dataset_class': dataset,
        'model_class': model_class,
        'resize_size': f'{resize} {resize}',
        'inference_dir_name': link_out,   # absoluto SEM espaços (via symlink) -> dict_to_argv ok
        'device': DEVICE,
        'use_amp': '',
        'force_headless': '',
        'skip_boxplot': '',
        'save_inference_images': '',      # salva as predições (PNG) em inference_results_<ds>/inferences/
    }
    if weights == 'vessshape':
        params['run_path'] = CHECKPOINTS[model_class]
        params['checkpoint_type'] = 'last'
    else:
        params['run_path'] = '.'            # placeholder: não lido quando skip_checkpoint_loading
        params['skip_checkpoint_loading'] = ''
        if weights == 'imagenet':
            params['encoder_weights'] = 'imagenet'
            if imagenet_normalize:
                params['imagenet_normalize'] = ''
        elif weights == 'litemedsam':
            params['channels'] = 'rgb'      # LiteMedSAM exige 3 canais
    # ResNet: channels omitido -> default do dataset (gray/all), casando com o few-shot.

    argv = [sys.executable, _TEST_PY] + dict_to_argv(params, _POSITIONAL)
    proc = subprocess.run(argv, cwd=_CWD, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f'test.py rc={proc.returncode}\nSTDERR tail:\n{proc.stderr[-1800:]}')
    return real_out


def run_many(variant_keys, datasets=None):
    """Roda um lote de variantes × datasets, capturando erros por item."""
    datasets = datasets or DATASETS_TO_RUN
    status = {}
    for vk in variant_keys:
        for ds in datasets:
            key = f'{vk} / {ds}'
            try:
                run_zero_shot(vk, ds)
                status[key] = 'OK'
                print(f'[OK]   {key}')
            except Exception as e:
                status[key] = f'FAIL: {e}'
                print(f'[FAIL] {key}: {e}')
    return status

## 1. VessShape (Zero-Shot VSUNet18 / VSUNet50)
Carrega os checkpoints pré-treinados em VessShape (`checkpoint_type='last'`) e infere sem fine-tuning.

In [4]:
status_vessshape = run_many(['resnet18_vessshape', 'resnet50_vessshape'])
status_vessshape

[OK]   resnet18_vessshape / vessmap


[OK]   resnet18_vessshape / drive


[OK]   resnet18_vessshape / dca1


[OK]   resnet18_vessshape / octa2d


[OK]   resnet50_vessshape / vessmap


[OK]   resnet50_vessshape / drive


[OK]   resnet50_vessshape / dca1


[OK]   resnet50_vessshape / octa2d


{'resnet18_vessshape / vessmap': 'OK',
 'resnet18_vessshape / drive': 'OK',
 'resnet18_vessshape / dca1': 'OK',
 'resnet18_vessshape / octa2d': 'OK',
 'resnet50_vessshape / vessmap': 'OK',
 'resnet50_vessshape / drive': 'OK',
 'resnet50_vessshape / dca1': 'OK',
 'resnet50_vessshape / octa2d': 'OK'}

## 2. ImageNet encoder (Zero-Shot IN-UNet18 / IN-UNet50)
`encoder_weights='imagenet'` + `skip_checkpoint_loading` — pesos do encoder vêm da `smp`; decoder aleatório. `IMAGENET_NORMALIZE` controla a normalização opcional dos inputs.

In [5]:
status_imagenet = run_many(['resnet18_imagenet', 'resnet50_imagenet'])
status_imagenet

[OK]   resnet18_imagenet / vessmap


[OK]   resnet18_imagenet / drive


[OK]   resnet18_imagenet / dca1


[OK]   resnet18_imagenet / octa2d


[OK]   resnet50_imagenet / vessmap


[OK]   resnet50_imagenet / drive


[OK]   resnet50_imagenet / dca1


[OK]   resnet50_imagenet / octa2d


{'resnet18_imagenet / vessmap': 'OK',
 'resnet18_imagenet / drive': 'OK',
 'resnet18_imagenet / dca1': 'OK',
 'resnet18_imagenet / octa2d': 'OK',
 'resnet50_imagenet / vessmap': 'OK',
 'resnet50_imagenet / drive': 'OK',
 'resnet50_imagenet / dca1': 'OK',
 'resnet50_imagenet / octa2d': 'OK'}

## 3. LiteMedSAM (Zero-Shot LiteMedSAM)
Carrega `lite_medsam.pth` interno (`skip_checkpoint_loading`), `channels='rgb'`, `resize 256`. **Atenção:** zero-shot puro (sem fine-tuning, prompt = caixa da imagem inteira) tende a Dice ~0 em vasos finos — é uma baseline legítima, não um bug.

In [6]:
status_litemedsam = run_many(['litemedsam'])
status_litemedsam

[OK]   litemedsam / vessmap


[OK]   litemedsam / drive


[OK]   litemedsam / dca1


[OK]   litemedsam / octa2d


{'litemedsam / vessmap': 'OK',
 'litemedsam / drive': 'OK',
 'litemedsam / dca1': 'OK',
 'litemedsam / octa2d': 'OK'}

## 4. Consolidação → `zero_shot/zero_shot_inference_results_on_<ds>.csv`
Lê a linha `mean` de cada `metrics_stats.csv` e monta o CSV consolidado por dataset (interface lida pelos `eval_*.ipynb`).

In [7]:
for ds in DATASETS_TO_RUN:
    df = build_zero_shot_csv(ds)
    labels = list(df['model_type']) if not df.empty else []
    print(f'{ds:8s} -> {len(df)} linha(s): {labels}')

vessmap  -> 5 linha(s): ['Zero-Shot VSUNet18', 'Zero-Shot VSUNet50', 'Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM']
drive    -> 5 linha(s): ['Zero-Shot VSUNet18', 'Zero-Shot VSUNet50', 'Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM']
dca1     -> 5 linha(s): ['Zero-Shot VSUNet18', 'Zero-Shot VSUNet50', 'Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM']
octa2d   -> 5 linha(s): ['Zero-Shot VSUNet18', 'Zero-Shot VSUNet50', 'Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM']


## 5. Sanidade / integração
Confirma que `load_dataset_results` capta as variantes zero-shot reais (sem `(mock)`).

In [8]:
for ds in DATASETS_TO_RUN:
    df = load_dataset_results(ds, root='..')
    zs = df[df['stage'] == 'zero_shot']
    flags = bool(zs['is_mock'].any()) if len(zs) else None
    print(f'{ds:8s} zero-shot: {sorted(zs["model_type"].unique())}  | is_mock any: {flags}')

vessmap  zero-shot: ['Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM', 'Zero-Shot VSUNet18', 'Zero-Shot VSUNet50']  | is_mock any: False


drive    zero-shot: ['Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM', 'Zero-Shot VSUNet18', 'Zero-Shot VSUNet50']  | is_mock any: False


dca1     zero-shot: ['Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM', 'Zero-Shot VSUNet18', 'Zero-Shot VSUNet50']  | is_mock any: False


octa2d   zero-shot: ['Zero-Shot IN-UNet18', 'Zero-Shot IN-UNet50', 'Zero-Shot LiteMedSAM', 'Zero-Shot VSUNet18', 'Zero-Shot VSUNet50']  | is_mock any: False


In [9]:
zs

,run_name,num_samples,run,rep,wandb_group,model_class,Accuracy,IoU,Precision,Recall,Dice,AUC,model_type,stage,experiment,is_mock,weights
1153,zero_shot_resnet18_vessshape,0,None,None,vessshape,resnet18_unet,0.906988,0.179097,0.703621,0.350771,0.272441,0.782416,Zero-Shot VSUNet18,zero_shot,__zero_shot_csv__,False,vessshape
1154,zero_shot_resnet50_vessshape,0,None,None,vessshape,resnet50_unet,0.923098,0.319489,0.713522,0.451371,0.469781,0.846473,Zero-Shot VSUNet50,zero_shot,__zero_shot_csv__,False,vessshape
1155,zero_shot_resnet18_imagenet,0,None,None,imagenet,resnet18_unet,0.863412,0.074710,0.153186,0.141695,0.138425,0.586518,Zero-Shot IN-UNet18,zero_shot,__zero_shot_csv__,False,imagenet
1156,zero_shot_resnet50_imagenet,0,None,None,imagenet,resnet50_unet,0.090859,0.082132,0.082208,0.988930,0.151281,0.447949,Zero-Shot IN-UNet50,zero_shot,__zero_shot_csv__,False,imagenet
1157,zero_shot_litemedsam,0,None,None,litemedsam,litemedsam,0.917757,0.000000,0.000000,0.000000,0.000000,0.526090,Zero-Shot LiteMedSAM,zero_shot,__zero_shot_csv__,False,litemedsam
